# Healthcare Admissions — Data Cleaning Pipeline

Cleans and validates a synthetic healthcare dataset (generated with [Synthea](https://synthetichealth.github.io/synthea/))
for use in a Power BI star-schema admissions dashboard.

**Pipeline steps per table:** load → profile → clean → validate → export.

**Design decisions**
- Dimension tables (`patients`, `providers`, `organizations`, `payers`) are cleaned and exported at **full scope** — no date filtering. These describe entities, not events, so trimming them by date would silently drop valid dimension members that a fact table might still reference.
- Fact / event tables (`encounters`, `conditions`, `procedures`, `claims`, `claims_transactions`) are cleaned at full scope first, then a **single, consistent** analysis window (`ANALYSIS_START`–`ANALYSIS_END`) is applied identically across all of them in the final section — so every fact table reflects the same reporting period.
- Encounter dates in this dataset are real (unshifted) Synthea timestamps, so calendar-year filtering and trending is valid here (unlike a MIMIC-style de-identified date-shifted dataset).


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Consistent reporting window applied to every fact table in the final section
ANALYSIS_START = pd.Timestamp("2020-01-01", tz="UTC")
ANALYSIS_END = pd.Timestamp("2022-12-31 23:59:59", tz="UTC")

def profile(df: pd.DataFrame, name: str) -> None:
    """One-line data quality snapshot: shape, duplicates, missingness."""
    missing = df.isna().sum().sum()
    print(f"{name:<20} rows={len(df):>9,}  cols={df.shape[1]:>3}  "
          f"duplicate_rows={df.duplicated().sum():>6,}  missing_values={missing:>9,}")


## 1. Patients (dimension)

In [ ]:
patients_raw = pd.read_csv(RAW_DATA / "patients.csv")
profile(patients_raw, "patients_raw")


In [ ]:
KEEP_COLS = [
    "Id", "BIRTHDATE", "DEATHDATE", "MARITAL", "RACE", "ETHNICITY", "GENDER",
    "CITY", "STATE", "COUNTY", "FIPS", "ZIP", "LAT", "LON",
    "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "INCOME",
]

patients_clean = patients_raw[KEEP_COLS].copy()

patients_clean["BIRTHDATE"] = pd.to_datetime(patients_clean["BIRTHDATE"], errors="coerce")
patients_clean["DEATHDATE"] = pd.to_datetime(patients_clean["DEATHDATE"], errors="coerce")
patients_clean["MARITAL"] = patients_clean["MARITAL"].fillna("Unknown")
patients_clean["IS_DECEASED"] = patients_clean["DEATHDATE"].notna()

# Age as of a fixed reference date. Ages that come out negative belong to
# patients born after the reference date (a real possibility with a rolling
# generation date) -- set to null rather than silently keeping a negative age.
REFERENCE_DATE = pd.Timestamp("2022-12-31")
age_years = (REFERENCE_DATE - patients_clean["BIRTHDATE"]).dt.days / 365.25
patients_clean["AGE"] = age_years.astype(int)
patients_clean.loc[patients_clean["AGE"] < 0, "AGE"] = pd.NA

patients_clean["AGE_GROUP"] = pd.cut(
    patients_clean["AGE"],
    bins=[0, 17, 34, 49, 64, 79, 120],
    labels=["0-17", "18-34", "35-49", "50-64", "65-79", "80+"],
    include_lowest=True,
)


In [ ]:
profile(patients_clean, "patients_clean")
assert patients_clean["Id"].is_unique, "Duplicate patient IDs found"
assert (patients_clean["AGE"].dropna() < 0).sum() == 0

patients_clean.to_csv(PROCESSED_DATA / "patients_clean.csv", index=False)
print("Saved patients_clean.csv")


## 2. Providers (dimension)

In [ ]:
providers_raw = pd.read_csv(RAW_DATA / "providers.csv")
profile(providers_raw, "providers_raw")

providers_clean = providers_raw.copy()

assert providers_clean["Id"].is_unique
assert ((providers_clean["LAT"].between(-90, 90)) & (providers_clean["LON"].between(-180, 180))).all()

providers_clean.to_csv(PROCESSED_DATA / "providers_clean.csv", index=False)
print("Saved providers_clean.csv")


## 3. Organizations (dimension)

In [ ]:
organizations_raw = pd.read_csv(RAW_DATA / "organizations.csv")
profile(organizations_raw, "organizations_raw")

organizations_clean = organizations_raw.copy()

assert organizations_clean["Id"].is_unique
assert (organizations_clean["REVENUE"] >= 0).all()
assert (organizations_clean["UTILIZATION"] >= 0).all()

organizations_clean.to_csv(PROCESSED_DATA / "organizations_clean.csv", index=False)
print("Saved organizations_clean.csv")


## 4. Payers (dimension)

In [ ]:
payers_raw = pd.read_csv(RAW_DATA / "payers.csv")
profile(payers_raw, "payers_raw")

payers_clean = payers_raw.copy()

assert payers_clean["Id"].is_unique
assert (payers_clean["AMOUNT_COVERED"] >= 0).all()
assert (payers_clean["AMOUNT_UNCOVERED"] >= 0).all()

payers_clean.to_csv(PROCESSED_DATA / "payers_clean.csv", index=False)
print("Saved payers_clean.csv")


## 5. Encounters (fact)

Cleaned at full scope with the derived columns the dashboard actually needs
(duration, standardized category, patient financial responsibility). The
2020-2022 analysis window is applied later, in Section 9, alongside every
other fact table -- not here in isolation.

In [ ]:
encounters_raw = pd.read_csv(RAW_DATA / "encounters.csv")
profile(encounters_raw, "encounters_raw")


In [ ]:
encounters_clean = encounters_raw.copy()

encounters_clean["START"] = pd.to_datetime(encounters_clean["START"], errors="coerce")
encounters_clean["STOP"] = pd.to_datetime(encounters_clean["STOP"], errors="coerce")

encounters_clean["ENCOUNTER_DURATION_HOURS"] = (
    (encounters_clean["STOP"] - encounters_clean["START"]).dt.total_seconds() / 3600
).round(2)

encounters_clean["ENCOUNTER_YEAR"] = encounters_clean["START"].dt.year
encounters_clean["ENCOUNTER_MONTH"] = encounters_clean["START"].dt.month
encounters_clean["ENCOUNTER_MONTH_NAME"] = encounters_clean["START"].dt.month_name()

ENCOUNTER_CATEGORY_MAP = {
    "ambulatory": "Outpatient", "outpatient": "Outpatient",
    "wellness": "Preventive",
    "urgentcare": "Urgent Care",
    "emergency": "Emergency",
    "inpatient": "Inpatient",
    "home": "Home Care",
    "snf": "Skilled Nursing",
    "virtual": "Telehealth",
    "hospice": "Hospice",
}
encounters_clean["ENCOUNTER_CATEGORY"] = encounters_clean["ENCOUNTERCLASS"].map(ENCOUNTER_CATEGORY_MAP)

encounters_clean["PATIENT_RESPONSIBILITY"] = (
    encounters_clean["TOTAL_CLAIM_COST"] - encounters_clean["PAYER_COVERAGE"]
).round(2)

encounters_clean["PAYER_COVERAGE_RATE"] = (
    encounters_clean["PAYER_COVERAGE"] / encounters_clean["TOTAL_CLAIM_COST"]
).round(4)


In [ ]:
profile(encounters_clean, "encounters_clean")
assert encounters_clean["Id"].is_unique
assert (encounters_clean["STOP"] >= encounters_clean["START"]).all()
assert encounters_clean["ENCOUNTER_CATEGORY"].notna().all(), "Unmapped ENCOUNTERCLASS value found"
assert (encounters_clean["TOTAL_CLAIM_COST"] >= 0).all()
assert (encounters_clean["PAYER_COVERAGE"] >= 0).all()

encounters_clean.to_csv(PROCESSED_DATA / "encounters_clean.csv", index=False)
print("Saved encounters_clean.csv")


## 6. Conditions (fact)

Clinical category assignment is a single keyword-plus-explicit-override map,
built once here rather than iteratively patched across many notebook cells.

In [ ]:
conditions_raw = pd.read_csv(RAW_DATA / "conditions.csv")
profile(conditions_raw, "conditions_raw")


In [ ]:
conditions_clean = conditions_raw.copy()

conditions_clean["START"] = pd.to_datetime(conditions_clean["START"], errors="coerce")
conditions_clean["STOP"] = pd.to_datetime(conditions_clean["STOP"], errors="coerce")

# Strip the trailing SNOMED classifier, e.g. "Chronic sinusitis (disorder)" -> "Chronic sinusitis"
conditions_clean["CONDITION_NAME"] = (
    conditions_clean["DESCRIPTION"].str.replace(r"\s*\([^)]*\)$", "", regex=True).str.strip()
)

CATEGORY_KEYWORDS = {
    "Dental": ["gingiv", "dental", "tooth", "teeth", "caries", "molar", "torus"],
    "Cardiovascular": ["myocardial infarction", "heart failure", "heart disease", "ischemic heart",
                        "hypertension", "coronary", "atrial fibrillation", "aortic valve",
                        "mitral valve", "pulmonic valve", "tricuspid valve", "pulmonary embolism",
                        "deep venous thrombosis", "preinfarction"],
    "Respiratory": ["sinusitis", "bronchitis", "pharyngitis", "pneumonia", "asthma", "otitis",
                     "wheezing", "sleep apnea", "emphysema", "hypoxemia", "rhinitis", "sore throat",
                     "nasal congestion", "sputum"],
    "Metabolic / Endocrine": ["diabetes", "obesity", "body mass index", "hyperlipidemia",
                               "hypertriglyceridemia", "hyperglycemia", "metabolic syndrome"],
    "Musculoskeletal / Injury": ["fracture", "sprain", "injury", "laceration", "scoliosis",
                                  "burn", "concussion", "bullet wound", "meniscus", "patellar",
                                  "osteoporosis", "fibromyalgia"],
    "Genitourinary": ["kidney", "renal", "cystitis", "urinary", "bladder", "pyelonephritis",
                       "retention of urine"],
    "Neurological": ["neurolog", "seizure", "migraine", "epilepsy", "alzheimer", "cerebral palsy",
                      "cerebrovascular", "stroke", "dementia", "spasticity", "intellectual disability",
                      "spina bifida"],
    "Women's Health": ["pregnan", "miscarriage", "prenatal", "postnatal", "postpartum",
                        "pre-eclampsia", "blighted ovum", "tubal ligation", "sterilization"],
    "Infectious Disease": ["infection", "infective", "viral", "bacterial", "coronavirus",
                            "streptococcal", "sepsis", "immune deficiency", "hepatitis"],
    "Mental / Behavioral": ["stress", "anxiety", "depress", "alcoholism", "drug abuse",
                             "opioid abuse", "misuses drugs", "smokes tobacco", "dependent drug"],
    "Oncology": ["cancer", "carcinoma", "leukemia", "lymphoma", "melanoma", "neoplasm", "myeloma"],
    "Hematologic": ["anemia", "coagulation disorder", "neutropenia"],
    "Gastrointestinal": ["appendic", "vomiting", "nausea", "gastro", "diarrhea", "bowel",
                          "bleeding from anus", "polyp of colon"],
    "Dermatological": ["dermatitis", "eczema", "rash", "allergic reaction"],
    "Symptoms": ["fever", "cough", "headache", "fatigue", "chill", "dyspnea", "pain",
                 "dystonia", "excessive salivation", "loss of taste", "shock"],
    "Social / SDOH": ["unemployed", "employment", "education", "social isolation",
                       "limited social contact", "intimate partner abuse", "not in labor force",
                       "homeless", "transport problem", "housing unsatisfactory", "refugee",
                       "criminal record", "military service", "risk activity", "social migrant",
                       "violence in the environment", "lack of access to transportation"],
    "Care / Administrative": ["medication review"],
    "Care / End-of-Life": ["hospice"],
}

def categorize_condition(name: str) -> str:
    lowered = name.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in lowered for kw in keywords):
            return category
    return "Unclassified"

conditions_clean["CLINICAL_CATEGORY"] = conditions_clean["CONDITION_NAME"].apply(categorize_condition)


In [ ]:
profile(conditions_clean, "conditions_clean")

unclassified_share = (conditions_clean["CLINICAL_CATEGORY"] == "Unclassified").mean()
print(f"Unclassified share: {unclassified_share:.2%}")
# Keep this comfortably low (a handful of rare condition names); revisit
# CATEGORY_KEYWORDS above if this creeps up after regenerating the population.

conditions_clean.to_csv(PROCESSED_DATA / "conditions_clean.csv", index=False)
print("Saved conditions_clean.csv")


## 7. Procedures (fact)

In [ ]:
procedures_raw = pd.read_csv(RAW_DATA / "procedures.csv")
profile(procedures_raw, "procedures_raw")

procedures_clean = procedures_raw.copy()
procedures_clean["START"] = pd.to_datetime(procedures_clean["START"], errors="coerce")
procedures_clean["STOP"] = pd.to_datetime(procedures_clean["STOP"], errors="coerce")

procedures_clean["PROCEDURE_DURATION_HOURS"] = (
    (procedures_clean["STOP"] - procedures_clean["START"]).dt.total_seconds() / 3600
)

# A handful of rows have STOP before START (data artifact) -- drop rather than clip,
# since a negative duration can't be trusted to represent the true procedure length.
before = len(procedures_clean)
procedures_clean = procedures_clean[procedures_clean["PROCEDURE_DURATION_HOURS"] >= 0].copy()
print(f"Dropped {before - len(procedures_clean)} row(s) with negative duration")


In [ ]:
profile(procedures_clean, "procedures_clean")
assert (procedures_clean["PROCEDURE_DURATION_HOURS"] >= 0).all()
assert (procedures_clean["BASE_COST"] >= 0).all()

procedures_clean.to_csv(PROCESSED_DATA / "procedures_clean.csv", index=False)
print("Saved procedures_clean.csv")


## 8. Claims & Claims Transactions (fact)

In [ ]:
claims_raw = pd.read_csv(RAW_DATA / "claims.csv")
profile(claims_raw, "claims_raw")

claims_clean = claims_raw.copy()

for col in ["CURRENTILLNESSDATE", "SERVICEDATE", "LASTBILLEDDATE1", "LASTBILLEDDATE2", "LASTBILLEDDATEP"]:
    claims_clean[col] = pd.to_datetime(claims_clean[col], errors="coerce")

for col in ["STATUS1", "STATUS2", "STATUSP"]:
    claims_clean[col] = claims_clean[col].str.strip().str.upper()

claims_clean["TOTAL_OUTSTANDING"] = (
    claims_clean["OUTSTANDING1"].fillna(0)
    + claims_clean["OUTSTANDING2"].fillna(0)
    + claims_clean["OUTSTANDINGP"].fillna(0)
).round(2)
claims_clean["HAS_OUTSTANDING_BALANCE"] = claims_clean["TOTAL_OUTSTANDING"] > 0

profile(claims_clean, "claims_clean")
assert claims_clean["Id"].is_unique
assert (claims_clean["TOTAL_OUTSTANDING"] >= 0).all()

claims_clean.to_csv(PROCESSED_DATA / "claims_clean.csv", index=False)
print("Saved claims_clean.csv")


In [ ]:
claims_txn_raw = pd.read_csv(RAW_DATA / "claims_transactions.csv")
profile(claims_txn_raw, "claims_transactions_raw")

claims_txn_clean = claims_txn_raw.copy()
claims_txn_clean["FROMDATE"] = pd.to_datetime(claims_txn_clean["FROMDATE"], errors="coerce")
claims_txn_clean["TODATE"] = pd.to_datetime(claims_txn_clean["TODATE"], errors="coerce")
claims_txn_clean["TYPE"] = claims_txn_clean["TYPE"].str.strip().str.upper()

for col in ["AMOUNT", "PAYMENTS", "ADJUSTMENTS", "TRANSFERS", "OUTSTANDING"]:
    claims_txn_clean[col] = pd.to_numeric(claims_txn_clean[col], errors="coerce").fillna(0).round(2)

profile(claims_txn_clean, "claims_transactions_clean")
assert claims_txn_clean["ID"].is_unique
assert (claims_txn_clean["AMOUNT"] >= 0).all()

claims_txn_clean.to_csv(PROCESSED_DATA / "claims_transactions_clean.csv", index=False)
print("Saved claims_transactions_clean.csv")


## 9. Consistent 2020-2022 analysis window

The full `_clean` tables above are the source of truth. Everything below
applies **the same** `ANALYSIS_START`/`ANALYSIS_END` window across every
fact table, so encounters, conditions, procedures, and claims all describe
the same reporting period in Power BI -- not four different windows.

In [ ]:
def apply_window(df: pd.DataFrame, date_col: str, name: str) -> pd.DataFrame:
    windowed = df[(df[date_col] >= ANALYSIS_START) & (df[date_col] <= ANALYSIS_END)].copy()
    print(f"{name:<22} full={len(df):>9,}  windowed={len(windowed):>9,}")
    return windowed

encounters_2020_2022 = apply_window(encounters_clean, "START", "encounters")
conditions_2020_2022 = apply_window(conditions_clean, "START", "conditions")
procedures_2020_2022 = apply_window(procedures_clean, "START", "procedures")
claims_2020_2022 = apply_window(claims_clean, "SERVICEDATE", "claims")
claims_txn_2020_2022 = apply_window(claims_txn_clean, "FROMDATE", "claims_transactions")


In [ ]:
encounters_2020_2022.to_csv(PROCESSED_DATA / "encounters_2020_2022.csv", index=False)
conditions_2020_2022.to_csv(PROCESSED_DATA / "conditions_2020_2022.csv", index=False)
procedures_2020_2022.to_csv(PROCESSED_DATA / "procedures_2020_2022.csv", index=False)
claims_2020_2022.to_csv(PROCESSED_DATA / "claims_2020_2022.csv", index=False)
claims_txn_2020_2022.to_csv(PROCESSED_DATA / "claims_transactions_2020_2022.csv", index=False)
print("Saved 5 windowed analysis tables to data/processed/")


## 10. Pipeline summary

In [ ]:
summary = pd.DataFrame([
    {"table": "patients_clean", "rows": len(patients_clean), "scope": "full"},
    {"table": "providers_clean", "rows": len(providers_clean), "scope": "full"},
    {"table": "organizations_clean", "rows": len(organizations_clean), "scope": "full"},
    {"table": "payers_clean", "rows": len(payers_clean), "scope": "full"},
    {"table": "encounters_clean", "rows": len(encounters_clean), "scope": "full"},
    {"table": "encounters_2020_2022", "rows": len(encounters_2020_2022), "scope": "windowed"},
    {"table": "conditions_clean", "rows": len(conditions_clean), "scope": "full"},
    {"table": "conditions_2020_2022", "rows": len(conditions_2020_2022), "scope": "windowed"},
    {"table": "procedures_clean", "rows": len(procedures_clean), "scope": "full"},
    {"table": "procedures_2020_2022", "rows": len(procedures_2020_2022), "scope": "windowed"},
    {"table": "claims_clean", "rows": len(claims_clean), "scope": "full"},
    {"table": "claims_2020_2022", "rows": len(claims_2020_2022), "scope": "windowed"},
    {"table": "claims_transactions_clean", "rows": len(claims_txn_clean), "scope": "full"},
    {"table": "claims_transactions_2020_2022", "rows": len(claims_txn_2020_2022), "scope": "windowed"},
])
summary
